# Quick Start

This page shows the **traditional first run** for CellPainting-Claw.

This page walks through one standard classical profiling run from input data to final profile outputs.

The run shown here does five concrete things in order:

- check the configured data source and stage a small demo download
- run CellProfiler extraction so the measurement tables are available
- merge those tables into one single-cell table
- use pycytominer to aggregate, annotate, normalize, and feature-select the classical profiles
- write summary tables and PCA views for quick inspection

DeepProfiler is not part of this page.

All output cells below are real recorded outputs from the current repository demo assets and current runtime.


## Install

From the repository root:


In [ ]:
conda env create -f environment/cellpainting-claw.environment.yml
conda activate cellpainting-claw
pip install -e .[data-access]


## Run Variables


In [ ]:
CONFIG=configs/project_config.demo.json
DATA_ROOT=demo/workspace/outputs/quick_start_data
RUN_ROOT=demo/workspace/outputs/quick_start_classical


## Input Staging

If your inputs still live in the Cell Painting Gallery, the normal entry sequence is:

1. inspect configured sources
2. plan a download
3. download a bounded slice into local storage

The commands below were run against the live Cell Painting Gallery with the demo config.


### Inspect Configured Sources


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-inspect-availability \
  --output-dir "$DATA_ROOT/01_inspect"


Default dataset: cpg0016-jump
Gallery datasets discovered: 42
Sources discovered under the default dataset: 14
Required data-access packages were available in the recorded runtime


Key files written here: `data_access_summary.json` and `pipeline_skill_manifest.json` under `$DATA_ROOT/01_inspect`.


### Download Plan


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-plan-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --max-files 4 \
  --output-dir "$DATA_ROOT/02_plan"


Resolved dataset: cpg0016-jump
Resolved source: source_4
Resolved Gallery prefix: cpg0016-jump/source_4/
Planned steps: 1
File cap in this example plan: 4


Key files written here: `download_plan.json` and `pipeline_skill_manifest.json` under `$DATA_ROOT/02_plan`.


### Download A Small Local Input Slice


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill data-download \
  --dataset-id cpg0016-jump \
  --source-id source_4 \
  --subprefix workspace/load_data_csv/2021_04_26_Batch1/BR00117035 \
  --output-dir "$DATA_ROOT/03_download_small"


Downloaded prefix: cpg0016-jump/source_4/workspace/load_data_csv/2021_04_26_Batch1/BR00117035/
Matched files: 2
Downloaded files: 2
Downloaded filenames: load_data.csv, load_data_with_illum.csv


Key files written here: `downloads/download_manifest.json`, plus `downloads/load_data.csv` and `downloads/load_data_with_illum.csv` under `$DATA_ROOT/03_download_small`.


This bounded download is only an input-staging example. The traditional profiling demo below uses the repository demo backend, which already includes a minimal local CellProfiler result set.


## Measurement Tables

The first classical step is to expose the CellProfiler measurement tables.


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-extract-measurements \
  --output-dir "$RUN_ROOT/01_measurements"


Demo mode: bundled measurement tables were reused
Exposed tables: Image.csv, Cells.csv, Nuclei.csv
Public skill entrypoint: cp-extract-measurements


Key files exposed here: `Image.csv`, `Cells.csv`, `Nuclei.csv`, and `pipeline_skill_manifest.json`. In the public demo checkout, these tables come from the bundled demo backend.


In this public demo checkout, the profiling backend script itself is not packaged. The skill therefore reuses the bundled demo measurement tables instead of rerunning CellProfiler. For user-owned workspaces, this same skill is still the public entrypoint for the measurement stage.


## Single-Cell Table


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cp-build-single-cell-table \
  --image-csv-path demo/backend/profiling_backend/outputs/cellprofiler/Image.csv \
  --object-table-path demo/backend/profiling_backend/outputs/cellprofiler/Cells.csv \
  --object-table Cells \
  --output-dir "$RUN_ROOT/02_single_cell"


Single-cell rows written: 4
Columns written: 16
Object table used for the merge: Cells


Key files written here: `single_cell.csv.gz` and `pipeline_skill_manifest.json` under `$RUN_ROOT/02_single_cell`.


This produces one merged single-cell table that pycytominer can consume directly.


## Classical Profiles


### Aggregate Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-aggregate-profiles \
  --single-cell-path "$RUN_ROOT/02_single_cell/single_cell.csv.gz" \
  --output-dir "$RUN_ROOT/03_cyto_aggregate"


Aggregated profile rows: 2
Aggregated profile columns: 14


Key files written here: `pycytominer/aggregated.parquet` and `pipeline_skill_manifest.json` under `$RUN_ROOT/03_cyto_aggregate`.


### Annotate Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-annotate-profiles \
  --aggregated-path "$RUN_ROOT/03_cyto_aggregate/pycytominer/aggregated.parquet" \
  --output-dir "$RUN_ROOT/04_cyto_annotate"


Annotated profile rows: 2
Annotated profile columns: 17


Key files written here: `pycytominer/annotated.parquet` and `pipeline_skill_manifest.json` under `$RUN_ROOT/04_cyto_annotate`.


### Normalize Profiles


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-normalize-profiles \
  --annotated-path "$RUN_ROOT/04_cyto_annotate/pycytominer/annotated.parquet" \
  --output-dir "$RUN_ROOT/05_cyto_normalize"


Normalized profile rows: 2
Normalized profile columns: 17


Key files written here: `pycytominer/normalized.parquet` and `pipeline_skill_manifest.json` under `$RUN_ROOT/05_cyto_normalize`.


### Select Profile Features


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-select-profile-features \
  --normalized-path "$RUN_ROOT/05_cyto_normalize/pycytominer/normalized.parquet" \
  --output-dir "$RUN_ROOT/06_cyto_select"


Feature-selected profile rows: 2
Feature-selected profile columns: 12


Key files written here: `pycytominer/feature_selected.parquet` and `pipeline_skill_manifest.json` under `$RUN_ROOT/06_cyto_select`.


These four pycytominer stages take the run from raw single-cell measurements to a smaller feature-selected well-level profile table.


## Summary Outputs


In [ ]:
cellpainting-skills run \
  --config "$CONFIG" \
  --skill cyto-summarize-classical-profiles \
  --feature-selected-path "$RUN_ROOT/06_cyto_select/pycytominer/feature_selected.parquet" \
  --output-dir "$RUN_ROOT/07_cyto_summary"


Summary rows represented: 2
Features retained at this stage: 6
Top variable features reported: 6
PCA components written: 2


Key files written here: `profile_summary.json`, `well_metadata_summary.csv`, `top_variable_features.csv`, `pca_coordinates.csv`, `pca_plot.png`, and `pipeline_skill_manifest.json` under `$RUN_ROOT/07_cyto_summary`.


This final step writes the first human-readable review layer for the classical profile branch.


## Result Files

After this Quick Start, the most useful files to inspect are:

- `data_access_summary.json` for the configured source inventory
- `download_plan.json` for the resolved Gallery request
- `single_cell.csv.gz` for the merged single-cell measurements table
- `aggregated.parquet`, `annotated.parquet`, `normalized.parquet`, and `feature_selected.parquet` for the pycytominer stages
- `profile_summary.json`, `well_metadata_summary.csv`, `top_variable_features.csv`, and `pca_plot.png` for the final review layer


## Next Pages

Continue with these pages after the first run:

- [CellPainting-Skills](../skills/index.md) for the full skill catalog
- [Command-Line Interface](../cli/index.md) for direct CLI usage
- [OpenClaw](../openclaw/index.md) for agent-mediated use of the same skills
